# Pipeline Zero-Shot NER — DisTemIST con Mistral-7B-Instruct-v0.3

Inferencia zero-shot para reconocimiento de enfermedades en textos clínicos en español.
El modelo base se carga directamente sin entrenamiento ni adaptadores LoRA.

## Contenido

1. [Entorno y dependencias](#1-entorno-y-dependencias)
2. [Configuración](#2-configuracion)
3. [Carga del modelo](#3-carga-del-modelo)
4. [Preparación para inferencia](#4-preparacion-para-inferencia)
5. [Inferencia](#5-inferencia)
6. [Evaluación](#6-evaluacion)


## 1. Entorno y dependencias

Instalación de paquetes e importación de librerías.

In [1]:
%%capture
!pip uninstall -y Pillow
!pip install "Pillow==11.3.0" --quiet

!pip install git+https://github.com/huggingface/transformers.git --upgrade --quiet

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

!pip install peft accelerate bitsandbytes trl==0.15.2 --quiet

!pip install -q spacy
!python -m spacy download es_core_news_md --quiet


## 2. Configuración

Hiperparámetros, rutas del dataset y credenciales.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
MAX_SEQ_LENGTH = 4096

# ── Dataset / chunking ────────────────────────────────────────────────────────
MAX_CHUNK_CHARS = None   # None para oraciones sueltas
SPACY_MODEL     = "es_core_news_md"

# ── Prompt del sistema ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Actua como un sistema NER medico de alta precision.\n\n"
    "REGLAS DE EXTRACCION:\n"
    "1. Extrae exclusivamente entidades de tipo ENFERMEDAD.\n"
    "2. COPIA Y PEGA de forma literal: No cambies mayusculas, minusculas ni tildes.\n"
    "3. PROHIBIDO USAR SINONIMOS: Si el texto dice 'neoplasia', no escribas 'cancer'.\n"
    "4. REPETICIONES: Si una enfermedad aparece varias veces en el texto, debes listarla varias veces en lineas separadas.\n"
    "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
    "6. FORMATO: Únicamente el texto de la mención, una por línea. PROHIBIDO usar caracteres de lista al inicio (como '-', '*', '•', '1.').\n"
    "7. Si no hay nada, devuelve un texto vacio."
)

# ── Inferencia ────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS   = 300
MAX_TEST_FILES   = 250   # None para procesar todos
PRINT_RAW_OUTPUT = True

# ── Rutas Distemist ───────────────────────────────────────────────────────────
PROJECT_ROOT         = "/kaggle/input/datasets/user"
DISTEMIST_ROOT       = f"{PROJECT_ROOT}/distemist/distemist"
TEXT_FILES_TEST_DIR  = f"{DISTEMIST_ROOT}/text_files_test"
GS_TEST_TSV          = f"{DISTEMIST_ROOT}/distemist_subtrack1_test_mentions.tsv"
PREDICTIONS_TSV      = "distemist_llm_predictions.tsv"
EVAL_SUMMARY_JSON    = "evaluation_summary.json"

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

print(f"Modelo base        : {MODEL_NAME}")
print(f"MAX_SEQ_LENGTH     : {MAX_SEQ_LENGTH}")
print(f"MAX_CHUNK_CHARS    : {MAX_CHUNK_CHARS}")
print(f"MAX_NEW_TOKENS     : {MAX_NEW_TOKENS}")
print(f"MAX_TEST_FILES     : {MAX_TEST_FILES}")


Modelo base        : unsloth/mistral-7b-instruct-v0.3-bnb-4bit
MAX_SEQ_LENGTH     : 4096
MAX_CHUNK_CHARS    : None
MAX_NEW_TOKENS     : 300
MAX_TEST_FILES     : 250


## 3. Carga del modelo

Carga cuantizada en 4-bit con distribución automática entre GPUs disponibles.

In [3]:
import torch
import unsloth
from unsloth import FastLanguageModel

print(f"GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB VRAM")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.float16,
    load_in_4bit   = True,
    token          = hf_token,
)

if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

print(f"\nModelo cargado: {MODEL_NAME}")
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
GPUs disponibles: 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM
==((====))==  Unsloth 2026.6.3: Fast Mistral patching. Transformers: 5.10.0.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

[unsloth_zoo.log|WARNING]Unsloth: Could not apply RoPE scaling 'default' from config (KeyError: 'default'); falling back to unscaled RoPE. Long-context generation may degrade.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]


Modelo cargado: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
Parámetros totales: 3,758,362,624


## 4. Preparación para inferencia

Carga de spaCy, configuración del pipeline generativo y listado de archivos de test.

In [4]:
import gc
import time
import re
import json
import logging
import numpy as np
import pandas as pd
import spacy
from pathlib import Path
from transformers import pipeline

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

nlp_spacy = spacy.load(SPACY_MODEL, disable=["ner", "lemmatizer"])
print(f"spaCy: {nlp_spacy.meta['name']} v{nlp_spacy.meta['version']}")


def chunk_text_by_sentences(text, nlp, max_chars=MAX_CHUNK_CHARS):
    doc = nlp(text)
    if max_chars is None:
        return [(text[s.start_char:s.end_char], s.start_char, s.end_char) for s in doc.sents]

    chunks, current_chars, chunk_start, current_end = [], 0, None, None
    for sent in doc.sents:
        if current_chars + len(sent.text) > max_chars and chunk_start is not None:
            chunks.append((text[chunk_start:current_end], chunk_start, current_end))
            current_chars, chunk_start, current_end = 0, None, None
        if chunk_start is None:
            chunk_start = sent.start_char
        current_chars += len(sent.text)
        current_end = sent.end_char
    if chunk_start is not None:
        chunks.append((text[chunk_start:current_end], chunk_start, current_end))
    return chunks


model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

def build_ner_messages(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Texto para analizar:\n{text}"},
    ]

txt_files = sorted(Path(TEXT_FILES_TEST_DIR).glob("*.txt"))
if MAX_TEST_FILES is not None:
    txt_files = txt_files[:MAX_TEST_FILES]

print(f"Archivos de test a procesar: {len(txt_files)}")


spaCy: core_news_md v3.8.0
Archivos de test a procesar: 250


## 5. Inferencia

Segmentación por oraciones, extracción de menciones y ajuste de offsets al documento completo.
Se guarda un checkpoint tras cada archivo para permitir reanudar la inferencia.

In [5]:
# ── Inferencia con checkpoint por archivo ─────────────────────────────────────
CHECKPOINT_FILE = "/kaggle/working/inference_checkpoint.json"

if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    pred_rows       = ckpt["pred_rows"]
    inference_times = ckpt["inference_times"]
    done_files      = set(ckpt["done_files"])
    print(f"Retomando desde checkpoint: {len(done_files)} archivos ya procesados")
else:
    pred_rows, inference_times, done_files = [], [], set()

total_files = len(txt_files)

for i, txt_path in enumerate(txt_files):
    if txt_path.stem in done_files:
        print(f"[{i+1}/{total_files}] {txt_path.name} — ya procesado, saltando")
        continue

    text         = txt_path.read_text(encoding="utf-8")
    raw_mentions = []
    t0           = time.time()

    chunks = chunk_text_by_sentences(text, nlp_spacy)

    for chunk_idx, (chunk_text, chunk_char_start, chunk_char_end) in enumerate(chunks):
        prompt = tokenizer.apply_chat_template(
            build_ner_messages(chunk_text),
            tokenize=False,
            add_generation_prompt=True,
        )
        try:
            result = pipe(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                max_length=None,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )[0]["generated_text"].strip()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                gc.collect()
                torch.cuda.empty_cache()
            continue

        if PRINT_RAW_OUTPUT:
            print(f"  [chunk {chunk_idx+1}/{len(chunks)} | chars {chunk_char_start}-{chunk_char_end}]")
            print(f"  RAW: {repr(result)}")

        search_cursors = {}
        for mention in [line.strip() for line in result.splitlines() if line.strip()]:
            pattern = re.escape(mention)
            cursor  = search_cursors.get(mention, chunk_char_start)
            match   = re.search(pattern, text[cursor:chunk_char_end], re.IGNORECASE)
            if not match:
                match = re.search(pattern, text[chunk_char_start:chunk_char_end], re.IGNORECASE)
                if not match:
                    continue
                off0 = chunk_char_start + match.start()
                off1 = chunk_char_start + match.end()
            else:
                off0 = cursor + match.start()
                off1 = cursor + match.end()
            search_cursors[mention] = off1
            raw_mentions.append((text[off0:off1], off0, off1))

    t_infer = time.time() - t0
    inference_times.append(t_infer)

    raw_mentions.sort(key=lambda x: x[1])
    seen_offsets = set()
    deduped      = []
    for span_text, off0, off1 in raw_mentions:
        if (off0, off1) not in seen_offsets:
            seen_offsets.add((off0, off1))
            deduped.append((span_text, off0, off1))

    for mark_idx, (span_text, off0, off1) in enumerate(deduped, 1):
        pred_rows.append({
            "filename": txt_path.stem,
            "mark":     f"T{mark_idx}",
            "label":    "ENFERMEDAD",
            "off0":     off0,
            "off1":     off1,
            "span":     span_text,
        })

    done_files.add(txt_path.stem)
    print(f"[{i+1}/{total_files}] {txt_path.name} | {t_infer:.2f}s | {len(chunks)} chunks | {len(deduped)} menciones")

    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"pred_rows": pred_rows, "inference_times": inference_times,
                   "done_files": list(done_files)}, f)

pd.DataFrame(pred_rows, columns=["filename", "mark", "label", "off0", "off1", "span"]).to_csv(
    PREDICTIONS_TSV, sep="\t", index=False
)
print(f"\nPredicciones guardadas en {PREDICTIONS_TSV}")
if inference_times:
    print(f"Tiempo medio por archivo: {sum(inference_times)/len(inference_times):.2f}s")


Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [chunk 1/13 | chars 0-90]
  RAW: 'Diabetes\nHipertensión\nObesidad'
  [chunk 2/13 | chars 91-170]
  RAW: 'Lumbar derecho\n\nTexto vacio'
  [chunk 3/13 | chars 171-204]
  RAW: 'Enfermedad no detectada en el texto.'
  [chunk 4/13 | chars 205-347]
  RAW: 'Suprarrenal derecha hipoecogénica\nneoplasia suprarrenal derecha'
  [chunk 5/13 | chars 348-472]
  RAW: 'Suprarrenal derecha - Enfermedad\nExpansivo - Enfermedad\nRiñón derecho - Enfermedad\nDesplazado hacia abajo - No es una enfermedad (Sin categorizar)'
  [chunk 6/13 | chars 473-667]
  RAW: 'Neoplasia suprarrenal\nTumor suprarrenal\nMasa tumoral\nNeoplasia de densidad grasa\nTumor de densidad grasa'
  [chunk 7/13 | chars 668-744]
  RAW: 'Cortisol normal\n\nFigura 1\n\nEl estudio hormonal para la determinación de prolactina fue anormalmente alto.\n\nEl paciente presenta una neoplasia maligna de la glándula tiroides.\n\nEl paciente presenta una neoplasia benigna de la glándula tiroides.\n\nEl paciente presenta una neoplasia maligna de 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [chunk 10/13 | chars 950-1003]
  RAW: 'Adrenalectomía derecha\nSe realizó adrenalectomía derecha sin complicaciones.\nNo se encontraron enfermedades en el texto.'
  [chunk 11/13 | chars 1004-1090]
  RAW: 'La enfermedad mencionada es: asintomática\nLa enfermedad mencionada es: post operatoria'
  [chunk 12/13 | chars 1090-1294]
  RAW: 'Enfermedad 1: Pieza con áreas amarillas - naranjas\nEnfermedad 2: Hemorragias\nEnfermedad 3: Tejido adiposo\nEnfermedad 4: Zonas de calcificaciones'
  [chunk 13/13 | chars 1295-1397]
  RAW: 'Mielolipoma\nAdrenal gland'
[1/250] S0004-06142006000100010-1.txt | 57.90s | 13 chunks | 10 menciones
  [chunk 1/21 | chars 0-197]
  RAW: 'adenocarcinoma gástrico\nhipertensión arterial'
  [chunk 2/21 | chars 198-412]
  RAW: 'Anorexia\nHipocrómica anemia\nEstreñimiento\nPérdida de peso (12 kilos en 6 meses)'
  [chunk 3/21 | chars 413-561]
  RAW: 'Gran tumoración\nneoplasia\nfirme\nindolora\nmóvil\nabdomen'
  [chunk 4/21 | chars 561-806]
  RAW: 'adenocarcinoma gástric

## 6. Evaluación

Métrica estricta por coincidencia exacta de offsets y métricas por solapamiento (IoU) a distintos umbrales.

In [6]:
import json
import pandas as pd


def prf(tp, fp, fn):
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


df_pred  = pd.read_csv(PREDICTIONS_TSV, sep="\t")
df_gs    = pd.read_csv(GS_TEST_TSV, sep="\t")
df_gs    = df_gs[df_gs["filename"].isin(df_pred["filename"].unique())]

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["off0"],   df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
p, r, f1   = prf(tp, fp, fn)

report = {"Estricta": {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 4), "recall": round(r, 4), "fscore": round(f1, 4)}}

print(f"── ESTRICTA ──  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  (TP={tp} FP={fp} FN={fn})\n")

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in df_pred["filename"].unique():
    gs_ints   = list(zip(df_gs[df_gs["filename"] == filename]["off0"],   df_gs[df_gs["filename"] == filename]["off1"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["off0"], df_pred[df_pred["filename"] == filename]["off1"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi, (p0, p1) in enumerate(pred_ints)
         for gi, (g0, g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp_t = len(matched_p)
        results[t]["tp"] += tp_t
        results[t]["fp"] += len(pred_ints) - tp_t
        results[t]["fn"] += len(gs_ints)   - tp_t

print("── IoU ──")
for t in thresholds:
    tp_t, fp_t, fn_t = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p_t, r_t, f1_t   = prf(tp_t, fp_t, fn_t)
    report[f"IoU >= {t}"] = {"tp": tp_t, "fp": fp_t, "fn": fn_t,
                              "precision": round(p_t, 4), "recall": round(r_t, 4), "fscore": round(f1_t, 4)}
    print(f"IoU >= {t}:  P={p_t:.4f}  R={r_t:.4f}  F1={f1_t:.4f}  (TP={tp_t} FP={fp_t} FN={fn_t})")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)


── ESTRICTA ──  P=0.2275  R=0.4039  F1=0.2911  (TP=1049 FP=3561 FN=1548)

── IoU ──
IoU >= 0.0:  P=0.3434  R=0.6095  F1=0.4393  (TP=1583 FP=3027 FN=1014)
IoU >= 0.5:  P=0.2859  R=0.5075  F1=0.3658  (TP=1318 FP=3292 FN=1279)
IoU >= 0.8:  P=0.2362  R=0.4193  F1=0.3022  (TP=1089 FP=3521 FN=1508)
